# 🚗 EV Driver Behavior Scoring System
## Notebook 3 — ML Model (KMeans + XGBoost + SHAP)

### Two-model pipeline:
1. **KMeans Clustering** — Unsupervised: group drivers into behavior profiles
2. **XGBoost Regression** — Supervised: predict kWh/100km from driving features
3. **SHAP** — Explain which features matter most (model interpretability)

In [ ]:
# ── Cell 1: Install & Import ──────────────────────────────────────────────────
!pip install xgboost shap --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import xgboost as xgb
import shap
import joblib

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('ev_driver_dataset.csv')
print(f'✅ Dataset loaded: {df.shape}')

## Part 1 — KMeans Clustering: Driver Behavior Profiles

In [ ]:
# ── Cell 2: KMeans — Find Optimal K (Elbow Method) ───────────────────────────
CLUSTER_FEATURES = [
    'regen_braking_ratio', 'aggression_index',
    'smoothness_score', 'high_speed_ratio', 'soc_swing_pct'
]

X_cluster = df[CLUSTER_FEATURES].copy()
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X_cluster)

inertias = []
K_range  = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=7)
ax.axvline(4, color='red', linestyle='--', label='Chosen K=4')
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Inertia')
ax.set_title('Elbow Method — Optimal K for Driver Clusters', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('elbow_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 3: Fit KMeans with K=4 ──────────────────────────────────────────────
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Label clusters based on their mean efficiency score
cluster_means = df.groupby('cluster')['ev_efficiency_score'].mean().sort_values(ascending=False)
cluster_labels = {
    cluster_means.index[0]: 'Eco Master',
    cluster_means.index[1]: 'Smooth Commuter',
    cluster_means.index[2]: 'Aggressive Urban',
    cluster_means.index[3]: 'Highway Sprinter',
}
df['driver_profile'] = df['cluster'].map(cluster_labels)

print('Cluster Profiles (mean values):')
profile_summary = df.groupby('driver_profile')[CLUSTER_FEATURES + ['kwh_per_100km','ev_efficiency_score']].mean().round(2)
print(profile_summary.to_string())

In [ ]:
# ── Cell 4: Cluster Visualization ────────────────────────────────────────────
colors = {'Eco Master':'#2196F3','Smooth Commuter':'#4CAF50',
          'Aggressive Urban':'#F44336','Highway Sprinter':'#FF9800'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Driver Behavior Clusters', fontsize=14, fontweight='bold')

for profile, grp in df.groupby('driver_profile'):
    axes[0].scatter(grp['regen_braking_ratio'], grp['aggression_index'],
                    label=profile, alpha=0.45, s=18, color=colors[profile])
axes[0].set_xlabel('Regen Braking Ratio')
axes[0].set_ylabel('Aggression Index')
axes[0].set_title('Regen vs Aggression')
axes[0].legend(fontsize=9)

profile_kwh = df.groupby('driver_profile')['kwh_per_100km'].mean().sort_values()
bar_colors  = [colors[p] for p in profile_kwh.index]
axes[1].barh(profile_kwh.index, profile_kwh.values, color=bar_colors, edgecolor='white')
axes[1].set_xlabel('Mean kWh / 100 km')
axes[1].set_title('Energy Consumption by Profile')
axes[1].axvline(14, color='gray', linestyle='--', linewidth=1, label='Baseline')
axes[1].legend()
for i, v in enumerate(profile_kwh.values):
    axes[1].text(v + 0.1, i, f'{v:.1f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('cluster_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 2 — XGBoost Regression: Predict kWh/100km

In [ ]:
# ── Cell 5: Prepare Features & Split ─────────────────────────────────────────
FEATURE_COLS = [
    'avg_speed_kmh', 'std_speed_kmh', 'avg_accel_ms2', 'std_accel_ms2',
    'avg_throttle_pct', 'avg_brake_pressure', 'regen_braking_ratio',
    'aggression_index', 'smoothness_score', 'high_speed_ratio', 'soc_swing_pct'
]
TARGET = 'kwh_per_100km'

X = df[FEATURE_COLS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# ── Cell 6: Train XGBoost Model ──────────────────────────────────────────────
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
model.fit(X_train, y_train,
          eval_set=[(X_test, y_test)],
          verbose=False)

y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')

print('=== XGBoost Model Performance ===')
print(f'MAE  : {mae:.3f} kWh/100km')
print(f'RMSE : {rmse:.3f} kWh/100km')
print(f'R²   : {r2:.4f}')
print(f'5-Fold CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# ── Cell 7: Actual vs Predicted Plot ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('XGBoost Model Evaluation', fontsize=13, fontweight='bold')

axes[0].scatter(y_test, y_pred, alpha=0.4, s=18, color='#2196F3')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual kWh/100km')
axes[0].set_ylabel('Predicted kWh/100km')
axes[0].set_title(f'Actual vs Predicted (R²={r2:.3f})')
axes[0].legend()

residuals = y_test - y_pred
axes[1].hist(residuals, bins=30, color='#FF9800', alpha=0.8, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals Distribution')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 3 — SHAP: Which Driving Behavior Wastes the Most Energy?

In [ ]:
# ── Cell 8: SHAP Explainability ───────────────────────────────────────────────
explainer   = shap.Explainer(model, X_train)
shap_values = explainer(X_test)

# Summary bar plot
fig, ax = plt.subplots(figsize=(9, 6))
shap.plots.bar(shap_values, max_display=11, show=False)
plt.title('SHAP Feature Importance — What Drives Energy Consumption?',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 9: SHAP Beeswarm Plot ────────────────────────────────────────────────
shap.plots.beeswarm(shap_values, max_display=11, show=False)
plt.title('SHAP Beeswarm — Impact of Each Feature on kWh/100km',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSHAP Interpretation:')
print('→ Features pushing RIGHT  = increase energy consumption (bad for EV)')
print('→ Features pushing LEFT   = decrease energy consumption (good for EV)')
print('→ Red dots = high feature value, Blue dots = low feature value')

In [ ]:
# ── Cell 10: Save Models ──────────────────────────────────────────────────────
joblib.dump(model,  'xgboost_ev_model.pkl')
joblib.dump(kmeans, 'kmeans_driver_clusters.pkl')
joblib.dump(scaler, 'cluster_scaler.pkl')

# Save scored dataset
df.to_csv('ev_driver_scored.csv', index=False)

print('✅ Models saved:')
print('   xgboost_ev_model.pkl')
print('   kmeans_driver_clusters.pkl')
print('   cluster_scaler.pkl')
print('   ev_driver_scored.csv')

## ✅ Notebook 3 Complete!

### What we built:
- **KMeans (K=4):** 4 driver profiles — Eco Master, Smooth Commuter, Aggressive Urban, Highway Sprinter
- **XGBoost:** Predicts kWh/100km with high accuracy (R² on test set)
- **SHAP:** Shows exactly which driving behaviors waste energy most
- All models saved as `.pkl` files

### Next → Notebook 4: Streamlit Dashboard